In [ ]:
import os
os.environ['PATH_TO_REPO'] = '/Users/stevie/repos/lingo_kit_super'

In [ ]:
path_to_repo = os.environ['PATH_TO_REPO']
path_to_repo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.DataFrame()
df_dir = os.path.join(path_to_repo, 'lingo_kit_data/vocabulary/sentences/short_lte50_sentences')
for file in os.listdir(df_dir):
    path = os.path.join(df_dir, file)
    temp = pd.read_csv(path, sep='\t')
    df = pd.concat([df, temp], ignore_index=True)
len(df), df.columns

In [ ]:
import sys
assert(os.path.exists(path_to_repo))
sys.path.append(path_to_repo)
from lingo_kit_data.utils.text_filter import classify_sentence

In [ ]:
from tqdm import tqdm

In [ ]:
for i, row in tqdm(df.iterrows(), total=len(df)):
    sentence = row['text_en']
    (is_appropriate, classification) = classify_sentence(sentence)
    df.loc[i, 'classification'] = classification

In [ ]:
df['classification'].value_counts()
# classification
# violence          2707
# swearing           497
# sexual content     217
# self-harm          152
# drugs               89
# hate speech         75

In [ ]:
violence_df = df[df['classification'] == 'violence']
swear_df = df[df['classification'] == 'swearing']
sexual_df = df[df['classification'] == 'sexual content']
selfharm_df = df[df['classification'] == 'self-harm']
drug_df = df[df['classification'] == 'drugs']
hate_df = df[df['classification'] == 'hate speech']

n = 5

print("Violence examples:")
for i in range(n):
    print(violence_df.iloc[i]['text_en'])

print("\nSwearing examples:")
for i in range(n):
    print(swear_df.iloc[i]['text_en'])

print("\nSexual content examples:")
for i in range(n):
    print(sexual_df.iloc[i]['text_en'])
    
print("\nSelf-harm examples:")
for i in range(n):
    print(selfharm_df.iloc[i]['text_en'])

print("\nDrug examples:")
for i in range(n):
    print(drug_df.iloc[i]['text_en'])

print("\nHate speech examples:")
for i in range(n):
    print(hate_df.iloc[i]['text_en'])

In [ ]:
# let's drop duplicates
print(len(df))
df = df.drop_duplicates(subset=['text_en', 'text_it'])
print(len(df))

In [ ]:
df = df[df['classification'].isnull()]
print(len(df))

In [ ]:
save_dir = os.path.join(
    path_to_repo,
    'lingo_kit_data/vocabulary/sentences/short_lte50_cleaned_sentences'
)
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# lets break this file into chunks
chunk_size = 100000
if len(df) % chunk_size == 0:
    n_chunks = len(df) // chunk_size
else: 
    n_chunks = len(df) // chunk_size + 1
for chunk_i in range(n_chunks):
    start_i = chunk_i * chunk_size
    end_i = min((chunk_i + 1) * chunk_size, len(df))
    chunk_df = df.iloc[start_i:end_i]
    save_path = os.path.join(save_dir, f'chunk_{chunk_i+1:02d}.tsv')
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    chunk_df.to_csv(save_path, sep='\t', index=False)